# Contoh Hitung Manual: Cold-to-Warm Threshold
Paper: *Recommendation Is a Dish Better Served Warm* (Gusak dkk., RecSys 2025)

Notebook ini memakai data mainan (6 user × 6 item) supaya **setiap langkah bisa dihitung tangan** dan dipindahkan ke slide:
1. Input
2. Preprocessing: split global timepoint
3. Model dasar ItemKNN (cosine similarity)
4. Output rekomendasi top-K
5. Metode paper, setup **User-Based**: potong riwayat jadi N → HR@K, NDCG@K
6. Deteksi threshold: sliding window least squares
7. Metode paper, setup **Item-Based**: batasi interaksi item jadi N → HR*@K, NDCG*@K

> Data mainan hanya untuk ilustrasi perhitungan, bukan replikasi hasil paper. Replikasinya ada di `02_ml1m_user_based.ipynb`.

In [1]:
import sys; sys.path.insert(0, "../src")
import numpy as np, pandas as pd
import coldwarm as cw
pd.set_option("display.precision", 3)
K = 2  # katalog hanya 6 item, jadi pakai top-2

## 1. Input
Data interaksi implisit (user menonton/membeli item) beserta timestamp. Item A–F, user U1–U6.

In [2]:
raw = [
    ("U1","A",1),("U1","B",2),("U1","C",3),("U1","D",4),
    ("U2","A",5),("U2","B",6),("U2","C",7),
    ("U3","B",8),("U3","C",9),("U3","E",10),
    ("U4","D",11),("U4","E",12),("U4","F",13),
    ("U5","A",14),("U5","B",15),("U5","D",16),("U5","C",30),("U5","E",32),
    ("U6","D",17),("U6","E",18),("U6","A",19),("U6","F",31),
]
df = pd.DataFrame(raw, columns=["user_id","item_id","timestamp"])
df, n_users, n_items = cw.encode(df)
U = dict(enumerate(pd.unique(df.user_id))); I = dict(enumerate(pd.unique(df.item_id)))
name = lambda items: [I[i] for i in items]
pd.crosstab(df.user_id, df.item_id)

item_id,A,B,C,D,E,F
user_id,,,,,,
U1,1,1,1,1,0,0
U2,1,1,1,0,0,0
U3,0,1,1,0,1,0
U4,0,0,0,1,1,1
U5,1,1,1,1,1,0
U6,1,0,0,1,1,1


## 2. Preprocessing: split global timepoint (Sec 4.1.2)
- GT = kuantil 0.9 dari semua timestamp.
- User yang punya interaksi **setelah** GT menjadi user test. Riwayatnya = interaksi sebelum GT.
- Untuk setup User-Based, ground truth = interaksi **pertama** setelah GT.
- Interaksi sebelum GT dipakai untuk training (data mainan tidak memakai set validasi).

In [3]:
split = cw.global_timepoint_split(df, q=0.9, val_frac=0.0)
print("GT =", split.gt_time)
for u in split.test_hist:
    print(f"{U[u]}: riwayat = {name(split.test_hist[u])}, holdout = {name(split.test_holdout[u])}, ground truth = {I[split.test_holdout[u][0]]}")
X = cw.to_matrix(split.train, n_users, n_items)
pd.DataFrame(X.toarray().astype(int), index=[U[u] for u in range(n_users)], columns=[I[i] for i in range(n_items)])

GT = 28.900000000000023
U5: riwayat = ['A', 'B', 'D'], holdout = ['C', 'E'], ground truth = C
U6: riwayat = ['D', 'E', 'A'], holdout = ['F'], ground truth = F


,A,B,C,D,E,F
U1,1,1,1,1,0,0
U2,1,1,1,0,0,0
U3,0,1,1,0,1,0
U4,0,0,0,1,1,1
U5,1,1,0,1,0,0
U6,1,0,0,1,1,0


## 3. Model dasar: ItemKNN
**Langkah 1:** matriks co-occurrence $G = X^\top X$. $G_{ab}$ = jumlah user yang berinteraksi dengan a **dan** b; diagonal = popularitas item.

In [4]:
G = (X.T @ X).toarray().astype(int)
pd.DataFrame(G, index=I.values(), columns=I.values())

,A,B,C,D,E,F
A,4,3,2,3,1,0
B,3,4,3,2,1,0
C,2,3,3,1,1,0
D,3,2,1,4,2,1
E,1,1,1,2,3,1
F,0,0,0,1,1,1


**Langkah 2:** cosine similarity $\text{sim}(a,b) = \dfrac{G_{ab}}{\sqrt{G_{aa}}\sqrt{G_{bb}}}$, diagonal = 0.

Contoh: $\text{sim}(A,B) = \dfrac{3}{\sqrt{4}\sqrt{4}} = 0.75$

In [5]:
model = cw.ItemKNN(k=None).fit(X)   # data mainan: tanpa pruning tetangga
S = pd.DataFrame(model.S, index=I.values(), columns=I.values())
S

,A,B,C,D,E,F
A,0.000,0.750,0.577,0.750,0.289,0.000
B,0.750,0.000,0.866,0.500,0.289,0.000
C,0.577,0.866,0.000,0.289,0.333,0.000
D,0.750,0.500,0.289,0.000,0.577,0.500
E,0.289,0.289,0.333,0.577,0.000,0.577
F,0.000,0.000,0.000,0.500,0.577,0.000


## 4. Output rekomendasi
$\text{skor}(u, j) = \sum_{h \in \text{riwayat}(u)} \text{sim}(h, j)$. Item yang sudah dilihat dibuang, lalu ambil top-K.

In [6]:
for u in split.test_hist:
    h = split.test_hist[u]
    sc = S.loc[name(h)].sum()
    print(f"{U[u]} riwayat {name(h)}")
    print("  skor:", sc.round(3).to_dict())
    rec = cw.recommend(model, [h], n_items, K)[0]
    print(f"  top-{K} (tanpa item yang sudah dilihat): {name(rec)}  | ground truth: {I[split.test_holdout[u][0]]}\n")

U5 riwayat ['A', 'B', 'D']
  skor: {'A': 1.5, 'B': 1.25, 'C': 1.7319999933242798, 'D': 1.25, 'E': 1.1549999713897705, 'F': 0.5}
  top-2 (tanpa item yang sudah dilihat): ['C', 'E']  | ground truth: C

U6 riwayat ['D', 'E', 'A']
  skor: {'A': 1.0390000343322754, 'B': 1.5390000343322754, 'C': 1.1990000009536743, 'D': 1.3270000219345093, 'E': 0.8659999966621399, 'F': 1.0770000219345093}
  top-2 (tanpa item yang sudah dilihat): ['B', 'C']  | ground truth: F



## 5. Setup User-Based (Sec 3.2)
Model **tetap** dan tidak dilatih ulang. Untuk tiap N:
1. ambil N interaksi secara acak (uniform) dari riwayat user, **urutkan sesuai waktu**
2. beri rekomendasi dari riwayat yang sudah dipotong
3. bandingkan dengan ground truth:

$$HR@K = \mathbb{1}[gt \in R^K_u], \qquad NDCG@K = \frac{\mathbb{1}[gt \in R^K_u]}{\log_2(\text{rank}+1)}$$

4. rata-rata atas user dengan riwayat ≥ N.

In [7]:
Ns = [1, 2, 3]
rng = np.random.default_rng(0)        # urutan acak sama dengan cw.user_based_curve(seed=0)
detail = []
for N in Ns:
    for u in split.test_hist:
        L = split.test_hist[u]
        if len(L) < N: continue
        idx = np.sort(rng.choice(len(L), size=N, replace=False))
        h = [L[j] for j in idx]
        rec = cw.recommend(model, [h], n_items, K)[0]
        gt = split.test_holdout[u][0]
        hit, ndcg = cw.hit_ndcg(rec[None, :], [gt])
        detail.append(dict(N=N, user=U[u], riwayat_N=name(h), rekomendasi=name(rec), gt=I[gt],
                           rank=(list(rec).index(gt)+1 if gt in rec else "-"),
                           HR=hit[0], NDCG=ndcg[0]))
detail = pd.DataFrame(detail); detail

,N,user,riwayat_N,rekomendasi,gt,rank,HR,NDCG
0,1,U5,[D],"[A, E]",C,-,0.0,0.000
1,1,U6,[E],"[D, F]",F,2,1.0,0.631
2,2,U5,"[A, B]","[C, D]",C,1,1.0,1.000
3,2,U6,"[D, A]","[B, C]",F,-,0.0,0.000
4,3,U5,"[A, B, D]","[C, E]",C,1,1.0,1.000
5,3,U6,"[D, E, A]","[B, C]",F,-,0.0,0.000


In [8]:
manual = detail.groupby("N")[["HR","NDCG"]].mean()
auto = cw.user_based_curve(model, split, n_items, Ns=Ns, K=K, seed=0).set_index("N")
print("Rata-rata per N (hitung manual):"); print(manual)
print("\nCek dengan fungsi modul:"); print(auto[["n_users","HR","NDCG"]])
assert np.allclose(manual.values, auto[["HR","NDCG"]].values)

Rata-rata per N (hitung manual):
    HR   NDCG
N            
1  0.5  0.315
2  0.5  0.500
3  0.5  0.500

Cek dengan fungsi modul:
   n_users   HR   NDCG
N                     
1        2  0.5  0.315
2        2  0.5  0.500
3        2  0.5  0.500


## 6. Deteksi threshold: sliding window least squares (Sec 4.2)
Untuk tiap window `w` titik berurutan pada kurva (N, metrik), hitung slope regresi linear:

$$a = \frac{\sum (N_k-\bar N)(y_k-\bar y)}{\sum (N_k-\bar N)^2}$$

Window dengan slope positif **tercuram** menandai transisi cold→warm. Threshold = N terakhir di window tersebut.
(Di data mainan hanya ada 3 titik, jadi w = 2. Lebar window tidak disebutkan di paper, jadi ini **asumsi**.)

In [9]:
thr, tab = cw.detect_threshold(manual.index, manual["NDCG"], window=2)
print(tab); print("\nThreshold (NDCG) =", thr)

   start  end      slope
0    1.0  2.0  1.845e-01
1    2.0  3.0  1.585e-16

Threshold (NDCG) = 2


## 7. Setup Item-Based (Sec 3.1)
Untuk item yang diteliti *i*:
1. $M(i)$ = interaksi item *i* di train. Interaksi user test dikeluarkan dari train.
2. ambil **N interaksi acak** dari $M(i)$, buang sisanya, lalu **latih ulang** model
3. item *i* dibuang dari riwayat dan holdout user test
4. **successive evaluation**: riwayat + holdout[:t] → top-K, untuk t = 0, 1, …
5. metrik modifikasi (Eq. 1–2), dihitung atas semua titik evaluasi:

$$HR^*@K(i) = \frac{1}{|U|}\sum_u \mathbb{1}[i \in R^K_u], \qquad NDCG^*@K(i) = \frac{1}{|U|}\sum_u \frac{\mathbb{1}[i\in R^K_u]}{\log_2(\text{rank}_u(i)+1)}$$

Contoh: item **B**.

In [10]:
split_ib = cw.global_timepoint_split(df, q=0.9, val_frac=0.0, include_test_history_in_train=False)
item = [k for k, v in I.items() if v == "B"][0]
train = split_ib.train
M = np.flatnonzero(train["i"].values == item)
print("M(B) =", [U[x] for x in train.iloc[M]["u"]])

eval_hists, who = [], []
for u in split_ib.test_hist:
    base = [x for x in split_ib.test_hist[u] if x != item]
    hold = [x for x in split_ib.test_holdout[u] if x != item]
    for t in range(len(hold)):
        eval_hists.append(base + hold[:t]); who.append(f"{U[u]} t={t}")
print("Titik evaluasi:", {w: name(h) for w, h in zip(who, eval_hists)})

rng = np.random.default_rng(0)
rows = []
for N in [1, 2, 3]:
    keep = rng.choice(M, size=N, replace=False)
    tau = train.drop(train.index[np.setdiff1d(M, keep)])
    m = cw.ItemKNN(k=None).fit(cw.to_matrix(tau, n_users, n_items))
    recs = cw.recommend(m, eval_hists, n_items, K)
    hr, nd = cw.modified_hit_ndcg(recs, item)
    rows.append(dict(N=N, B_disimpan_dari=[U[x] for x in train.loc[train.index[keep], "u"]],
                     rekomendasi={w: name(r) for w, r in zip(who, recs)},
                     **{f"HR*@{K}": round(hr, 3), f"NDCG*@{K}": round(nd, 3)}))
pd.DataFrame(rows)

M(B) = ['U1', 'U2', 'U3']
Titik evaluasi: {'U5 t=0': ['A', 'D'], 'U5 t=1': ['A', 'D', 'C'], 'U6 t=0': ['D', 'E', 'A']}


,N,B_disimpan_dari,rekomendasi,HR*@2,NDCG*@2
0,1,[U3],"{'U5 t=0': ['C', 'F'], 'U5 t=1': ['E', 'F'], '...",0.000,0.000
1,2,"[U3, U2]","{'U5 t=0': ['C', 'F'], 'U5 t=1': ['B', 'E'], '...",0.333,0.333
2,3,"[U2, U3, U1]","{'U5 t=0': ['B', 'C'], 'U5 t=1': ['B', 'E'], '...",1.000,1.000


**Cara membaca:** semakin banyak interaksi B yang diketahui model saat training, semakin sering B masuk top-K (HR\*) dan semakin tinggi posisinya (NDCG\*). N tempat kenaikan paling tajam terjadi adalah threshold cold→warm untuk item. Di data asli, nilai ini dirata-rata atas banyak item (Eq. 3–4) lalu dicari dengan sliding window yang sama seperti di bagian 6.